In [ ]:
# !pip install google-api-python-client pandas openpyxl -q

In [ ]:
import re
import time
import pandas as pd
from googleapiclient.discovery import build

In [ ]:
import os

# =========================
# 1. YouTube API 설정
# =========================
API_KEY = os.environ["YOUTUBE_API_KEY"]
youtube = build("youtube", "v3", developerKey=API_KEY)

In [ ]:
# =========================
# 2. 리버풀 관련 유튜브 영상 URL 목록
# =========================
video_urls = [
    "https://www.youtube.com/watch?v=Enu8Lc0zdkc",
    "https://www.youtube.com/watch?v=bytZAH_22Y8",
    "https://www.youtube.com/watch?v=x2cU8_DFCXk",
    "https://www.youtube.com/watch?v=wTcs3SJI4Pk",
    "https://www.youtube.com/watch?v=NfbT25bl8m4",
    "https://www.youtube.com/watch?v=rYcDLdgw0ro",
    "https://www.youtube.com/watch?v=K8vzefYXqX8",
    "https://www.youtube.com/watch?v=0unWeAwsHps",
    "https://www.youtube.com/watch?v=N_F68ZEwmDQ",
    "https://www.youtube.com/watch?v=VLKKBhXNRwE",
]

MAX_COMMENTS_PER_VIDEO = 1000

def extract_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([A-Za-z0-9_-]{11})", url)
    return match.group(1) if match else None

def clean_comment(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:

# =========================
# 3. 댓글 수집
# =========================
all_comments = []
seen_comment_ids = set()

print("수집할 영상 개수:", len(video_urls))
print("=" * 60)

for idx, url in enumerate(video_urls, start=1):
    video_id = extract_video_id(url)
    if not video_id:
        print(f"[건너뜀] video_id를 추출할 수 없음: {url}")
        continue

    print(f"\n[{idx}/{len(video_urls)}] 댓글 수집 시작: {video_id}")
    page_count = 0
    video_comment_count = 0
    next_page_token = None

    try:
        while True:
            request = youtube.commentThreads().list(part="snippet", videoId=video_id, maxResults=100, textFormat="plainText", order="relevance", pageToken=next_page_token)
            response = request.execute()
            page_count += 1
            items = response.get("items", [])
            new_count_this_page = 0

            for item in items:
                comment = item["snippet"]["topLevelComment"]
                comment_id = comment["id"]
                if comment_id in seen_comment_ids:
                    continue
                seen_comment_ids.add(comment_id)
                comment_data = comment["snippet"]
                original_comment = comment_data.get("textDisplay", "")
                clean_text = clean_comment(original_comment)
                all_comments.append({"comment_id": comment_id, "video_id": video_id, "video_url": url, "author": comment_data.get("authorDisplayName"), "comment_original": original_comment, "comment_clean": clean_text, "like_count": comment_data.get("likeCount"), "published_at": comment_data.get("publishedAt")})
                video_comment_count += 1
                new_count_this_page += 1

            print(f"[진행중] 영상 {video_id} | {page_count}페이지 완료 | 이번 페이지 신규 {new_count_this_page}개 | 현재 {video_comment_count}개 수집")

            if new_count_this_page == 0:
                print(f"[중단] 영상 {video_id} | 신규 댓글 0개라 반복 수집 방지 중단")
                break
            if video_comment_count >= MAX_COMMENTS_PER_VIDEO:
                print(f"[중단] 영상 {video_id} | {MAX_COMMENTS_PER_VIDEO}개 도달")
                break
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
            time.sleep(0.1)
        print(f"[완료] 영상 {video_id} | 총 {video_comment_count}개 수집")
    except Exception as e:
        print(f"[에러 발생] 영상 {video_id}")
        print(e)
        continue

In [ ]:
# =========================
# 4. 데이터프레임 생성 및 저장
# =========================
df = pd.DataFrame(all_comments)
print("\n" + "=" * 60)
print("전체 원본 수집 댓글 수:", len(df))

if len(df) > 0:
    df = df[df["comment_clean"].str.len() > 0].copy()
    df = df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
    print("중복 제거 후 댓글 수:", len(df))
    df.to_csv("liverpool_youtube_comments_clean.csv", index=False, encoding="utf-8-sig")
    df.to_excel("liverpool_youtube_comments_clean.xlsx", index=False)
    print("저장 완료")
    print("CSV 파일: liverpool_youtube_comments_clean.csv")
    print("Excel 파일: liverpool_youtube_comments_clean.xlsx")
else:
    print("수집된 댓글이 없습니다.")

df.head()


## GPT API 1차 감정분석


In [ ]:
# !pip install openai tqdm

In [ ]:
import pandas as pd
import json
import time
from tqdm import tqdm
from openai import OpenAI

client = OpenAI()

# =========================
# 1. 댓글 파일 불러오기
# =========================
df = pd.read_csv("liverpool_youtube_comments_clean.csv")
print("전체 댓글 수:", len(df))
df.head()


In [ ]:
# =========================
# 2. 감정분석 함수
# =========================
def analyze_sentiment(comment):
    prompt = f"""
너는 한국어 축구 팬덤 댓글을 분석하는 감정분석 분류기다.
분석 대상은 EPL 리버풀 FC 관련 유튜브 댓글이다.

분석의 목표는 댓글이 리버풀, 리버풀 선수, 리버풀 감독, 경기력, 우승 경쟁, 이적/전술, 팬덤 분위기에 대해 긍정적인지, 부정적인지, 중립적인지 분류하는 것이다.

[분류 기준]

1. positive
다음 중 하나에 해당하면 positive로 분류한다.
- 리버풀 팀, 감독, 선수, 전술, 경기 결과, 분위기를 긍정적으로 평가함
- 리버풀 선수의 실력, 폼, 득점, 도움, 수비, 압박, 헌신, 리더십을 칭찬함
- 리버풀의 승리, 우승, 순위 경쟁, 챔피언스리그 진출, 미래 가능성을 긍정적으로 언급함
- 농담이나 비속어가 있어도 실제 의미가 리버풀 칭찬이면 positive로 판단함
- 단, 리버풀 선수의 실수, 넘어짐, 미끄러짐, 부진을 웃음거리로 만드는 농담은 negative로 분류한다.

예시:
- "리버풀 진짜 미쳤다"
- "살라 폼 아직 살아있네"
- "반다이크 수비는 클래스가 다르다"
- "클롭 이후에도 팀이 잘 굴러간다"
- "이번 시즌 우승 가능성 있다"
- "리버풀 경기력 개좋다"

2. negative
다음 중 하나에 해당하면 negative로 분류한다.
- 리버풀 팀, 감독, 선수, 전술, 경기 결과, 분위기를 부정적으로 평가함
- 리버풀 선수의 실수, 부진, 결정력 부족, 수비 불안, 태도, 경기력을 비판함
- 리버풀의 패배, 우승 실패, 순위 하락, 답답한 경기, 이적 실패를 부정적으로 언급함
- 조롱, 비난, 실망, 분노가 리버풀을 향해 있으면 negative로 판단함

예시:
- "리버풀 오늘 왜 이렇게 못하냐"
- "각포 결정력 진짜 답답하다"
- "수비가 너무 불안하다"
- "전술이 너무 단조롭다"
- "이러면 우승 못하지"
- "중요한 경기에서 또 미끄러지네"

3. neutral
다음 중 하나에 해당하면 neutral로 분류한다.
- 단순 정보, 질문, 경기 일정, 선수 이름 언급만 있음
- 감정 방향이 명확하지 않음
- 댓글이 너무 짧거나 맥락이 부족함
- 농담, 밈, 감탄사만 있고 리버풀에 대한 긍정/부정 판단이 어려움
- 다른 댓글에 대한 반응처럼 보임
- 리버풀이 아닌 국가대표, 다른 구단, 심판, 리그 전체가 중심인 댓글
- 리버풀 팬인지 상대 팬인지 불명확하고 리버풀 관련 감정 판단이 어려움

예시:
- "오늘 경기 몇 시임?"
- "살라 재계약하나요?"
- "ㅋㅋㅋㅋㅋㅋ"
- "이건 좀 애매한데"
- "맨시티도 강하긴 하다"
- "프리미어리그 재밌다"

[리버풀 팬덤 멸칭 및 조롱 표현 규칙]

다음 표현은 리버풀을 조롱하거나 비하하는 표현으로 간주한다.

아래 단어가 포함되면 문맥과 관계없이 sentiment를 negative로 분류한다.

- 리중딱
- 살인풀
- 전범풀
- 뎀바바
- 소라드

또한 "훔" 또는 "훔바"로 시작하는 단어가 포함되면 sentiment를 negative로 분류한다.

예시:
- 리중딱
- 살인풀 또 시작이네
- 전범풀 수준
- 뎀바바는 영원한 레전드
- 소라드 또 날렸네
- 훔바훔바
- 훔○○

위 표현들은 리버풀에 대한 조롱, 비하, 멸칭으로 간주하며 반드시 negative로 분류한다.

[중요한 축구 팬덤 표현 해석]
다음 표현들은 문맥에 따라 판단한다.
- "미쳤다", "지린다", "개쩐다", "폼 미쳤다", "잘한다", "클래스", "GOAT", "레전드"가 칭찬 맥락이면 positive
- "개못한다", "답답하다", "망했다", "처참하다", "최악", "왜 저러냐"가 비판 맥락이면 negative
- "ㅋㅋ", "ㄷㄷ", "와", "개웃기네"만으로는 감정 판단이 어려우면 neutral
- 비속어가 포함되어도 대상이 리버풀 칭찬이면 positive, 리버풀 비판이면 negative
- 상대 팀이나 상대 선수를 칭찬하는 댓글은 리버풀에 대한 직접 평가가 아니면 neutral로 분류한다.
- 단, 상대 팀/선수 칭찬이 리버풀의 약점, 패배, 부진을 강조하는 맥락이면 negative로 분류한다.
- 리버풀을 조롱하거나 비꼬는 댓글은 negative로 분류한다.
- 리버풀 팬의 실망, 불안, 분노는 negative로 본다.
- 리버풀 우승, 명승부, 역전승, 안필드 분위기, YNWA, 레전드 선수 언급이 긍정 맥락이면 positive로 본다.

[판단 우선순위]
1. 댓글의 감정 대상이 리버풀 팀/선수/감독/경기력인지 확인한다.
2. 대상이 리버풀이면 긍정/부정/중립을 판단한다.
3. 대상이 리버풀이 아니거나 불명확하면 neutral로 둔다.
4. 단, 축구 커뮤니티에서 일반적으로 사용되는 칭찬 표현 또는 비판 표현이 존재하면 neutral보다 positive 또는 negative를 우선 선택한다.
5. neutral은 정말로 감정 방향을 판단할 수 없는 경우에만 사용한다.

[neutral 최소화 규칙]

다음 경우는 neutral이 아니다.

- 선수 이름 + 칭찬 표현 → positive
- 선수 이름 + 비판 표현 → negative
- 감독 이름 + 칭찬 표현 → positive
- 감독 이름 + 비판 표현 → negative
- 경기력 언급 + 긍정 평가 → positive
- 경기력 언급 + 부정 평가 → negative
- 우승 가능성, 좋은 순위, 좋은 경기 결과 → positive
- 패배, 부진, 실망, 우승 실패 → negative

neutral은 아래 경우에만 사용한다.

- 경기 시간 질문
- 선수 정보 질문
- 단순 사실 전달
- 의미 없는 감탄사
- 맥락 파악이 불가능한 짧은 댓글


[소보슬라이(소보슬러이) 조롱 표현 규칙]

다음 조건에 해당하면 sentiment를 negative로 분류한다.

- 소보슬라이 또는 소보슬러이를 제라드와 비교하며 넘어짐, 미끄러짐, 실수 장면을 조롱하는 경우
- 소보슬라이 또는 소보슬러이를 제라드의 유명한 미끄러짐 장면과 연결하여 비하하는 경우
- 소보슬라이 또는 소보슬러이의 실수, 넘어짐, 미끄러짐을 웃음거리로 만드는 경우
- "제라드 후계자", "제2의 제라드", "제라드 빙의", "제라드 DNA", "제라드 환생" 등의 표현이 조롱 의도로 사용되는 경우
- "미끄덩", "또 넘어지네", "슬라이딩", "넘어져서 골 먹힘" 등과 함께 소보슬라이(소보슬러이)가 언급되는 경우

예시:sh

- 소보슬라이 제라드 후계자네 ㅋㅋ
- 소보슬러이도 결국 제라드 과학
- 제2의 제라드 탄생
- 또 미끄러졌네
- 제라드 DNA 어디 안 간다
- 소보슬라이 넘어지는 거 개웃기네
- 소보슬러이 미끄덩 ㅋㅋ
- 제라드 환생했네

위와 같은 표현은 리버풀 선수에 대한 조롱 및 비하로 간주하며 반드시 negative로 분류한다.


[조롱성 웃음 표현 규칙]

댓글에 "ㅋㅋ", "ㅋㅋㅋ", "개웃기네", "존나 웃기네", "웃음벨", "개꿀잼", "레전드", "과학" 등이 포함되어 있더라도

리버풀 또는 리버풀 선수의 실수, 패배, 미끄러짐, 부진, 실책을 비웃는 맥락이면 반드시 negative로 분류한다.

예시:

- 또 넘어졌네 ㅋㅋㅋ
- 소보슬라이 과학이네
- 살라 폼 다 죽었네 ㅋㅋ
- 리버풀 또 졌네 개웃기네

[target 분류 기준]
- liverpool_team: 리버풀 팀 전체, 구단, 팬덤, 우승, 순위, 경기 결과에 대한 평가
- liverpool_player: 살라, 반다이크, 알리송, 비르츠, 에키티케, 각포 등 리버풀 선수 개인에 대한 평가
- liverpool_manager: 리버풀 감독, 코칭스태프, 전술 운영에 대한 평가
- opponent: 상대 팀이나 상대 선수 중심의 평가
- referee_or_league: 심판, 판정, EPL/챔스 등 대회나 리그 운영 중심의 평가
- unclear: 대상이 불명확하거나 리버풀과 직접 관련이 약한 경우

positive 예시 추가

- 살라 아직 클래스 살아있다
- 반다이크 진짜 GOAT
- 슬롯 전술 좋다
- 이번 경기 경기력 좋았다
- 리버풀 다시 올라온다
- YNWA

negative 예시 추가

- 살라 폼 다 죽었네
- 슬롯 경질해야 한다
- 전술이 답이 없다
- 이번 시즌 망했다
- 경기력 너무 처참하다
- 우승은 글렀다
- 개노잼 경기다

[출력 규칙]
반드시 아래 JSON 형식으로만 답하라.
설명 문장, 코드블록, 마크다운은 쓰지 마라.
{{
  "sentiment": "positive 또는 negative 또는 neutral",
  "confidence": 0.0,
  "target": "liverpool_team 또는 liverpool_player 또는 liverpool_manager 또는 opponent 또는 referee_or_league 또는 unclear",
  "reason": "짧은 한국어 이유"
  
}}

[분석할 댓글]
{comment}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "너는 한국어 리버풀 유튜브 댓글 감정분석 전문가다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        result_text = response.choices[0].message.content.strip()
        result = json.loads(result_text)
        return (result.get("sentiment", "error"), result.get("confidence", 0.0), result.get("target", "unclear"), result.get("reason", ""))
    except Exception as e:
        return "error", 0.0, "error", str(e)


In [ ]:
sentiments = []
confidences = []
targets = []
reasons = []

for comment in tqdm(df["comment_clean"]):
    sentiment, confidence, target, reason = analyze_sentiment(comment)
    sentiments.append(sentiment)
    confidences.append(confidence)
    targets.append(target)
    reasons.append(reason)
    time.sleep(0.2)

df["gpt_sentiment"] = sentiments
df["gpt_confidence"] = confidences
df["gpt_target"] = targets
df["gpt_reason"] = reasons

df[["comment_clean", "gpt_sentiment", "gpt_confidence", "gpt_target", "gpt_reason"]]


In [ ]:
df.to_csv("liverpool_youtube_comments_clean_감정분석_v2.csv", index=False, encoding="utf-8-sig")
df.to_excel("liverpool_youtube_comments_clean_감정분석_v2.xlsx", index=False)